<a href="https://colab.research.google.com/github/leonixxxxx/AI-Detect-Experiment/blob/main/Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# 1. INSTALL DEPENDENCIES & IMPORT LIBRARIES
# ==========================================
!pip install datasets transformers torch torchvision tqdm -q

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from PIL import Image

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# ==========================================
# 2. LOAD & PREPARE THE DATASET
# ==========================================
print("Downloading Tiny-GenImage dataset from Hugging Face...")
# This is a scaled-down dataset perfect for testing.
# Label 0 = Real, Label 1 = Fake (AI Generated)
dataset = load_dataset("TheKernel01/Tiny-GenImage")

# Define Data Augmentations (This is where you simulate LINE)
# We apply standard resizing for the model, plus JPEG compression and resizing
# to simulate how LINE degrades image quality.
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    # Optional: You can simulate LINE screenshot cropping here later
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom Dataset class to apply transforms
class GenImageDataset(Dataset):
    def __init__(self, hf_dataset, transform):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image'].convert('RGB') # Ensure 3 color channels
        label = item['label']

        if self.transform:
            image = self.transform(image)

        return image, label

# Create PyTorch DataLoaders
# We use a small subset of the training data just to prove the code works quickly.
# Remove `.select(range(2000))` to train on the full dataset.
train_data = GenImageDataset(dataset['train'].select(range(2000)), train_transforms)
val_data = GenImageDataset(dataset['validation'].select(range(500)), val_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=2)

README.md:   0%|          | 0.00/4.35k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  477MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  475MB            

data/train-00001-of-00014.parquet: downloading bytes:           |  0.00B            

...

data/validation-00002-of-00004.parquet: downloading bytes:           |  0.00B            

data/validation-00003-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  425MB            

data/validation-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/28000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [3]:
# ==========================================
# 3. BUILD THE MODEL (ResNet50 Transfer Learning)
# ==========================================
print("Loading Pretrained ResNet50...")
# Start with ImageNet weights
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze early layers to speed up training and prevent forgetting general features
for param in model.parameters():
    param.requires_grad = False

# Replace the final classification layer (fc) for our 2 classes: Real (0) vs AI (1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model = model.to(device)

# Define Loss function and Optimizer
# We only optimize the parameters of the new final layer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

Loading Pretrained ResNet50...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 75.0MB/s]


In [4]:

# ==========================================
# 4. TRAINING LOOP
# ==========================================
num_epochs = 3 # Keep it small for the baseline test

print("Starting Training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Training
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        train_bar.set_postfix(loss=loss.item(), acc=100.*correct/total)

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100. * val_correct / val_total
    print(f"End of Epoch {epoch+1} - Val Accuracy: {val_acc:.2f}%\n")

# ==========================================
# 5. SAVE THE MODEL
# ==========================================
torch.save(model.state_dict(), 'baseline_ai_detector.pth')
print("Model saved as 'baseline_ai_detector.pth'")

Starting Training...


Epoch 1/3 [Val]: 100%|██████████| 16/16 [02:18<00:00,  8.64s/it]


End of Epoch 1 - Val Accuracy: 75.80%



Epoch 2/3 [Val]: 100%|██████████| 16/16 [02:28<00:00,  9.27s/it]


End of Epoch 2 - Val Accuracy: 75.20%



Epoch 3/3 [Val]: 100%|██████████| 16/16 [02:34<00:00,  9.66s/it]


End of Epoch 3 - Val Accuracy: 76.60%

Model saved as 'baseline_ai_detector.pth'


### Run the LINE Simulation & Evaluation Code

In [5]:
# ==========================================
# 1. DEFINE LINE DEGRADATION SIMULATOR
# ==========================================
import io
from PIL import Image
import torch
from torchvision import transforms, models
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

# This function simulates what LINE does to an image in memory
class SimulateLINECompression(object):
    def __init__(self, quality=70, max_size=1080):
        self.quality = quality
        self.max_size = max_size

    def __call__(self, img):
        # 1. Resize simulating LINE's maximum dimension limits
        img.thumbnail((self.max_size, self.max_size), Image.Resampling.LANCZOS)

        # 2. Simulate JPEG compression artifacting
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=self.quality)
        buffer.seek(0)

        # 3. Reload the degraded image
        img_degraded = Image.open(buffer).convert('RGB')
        return img_degraded

# Create two validation pipelines: Clean vs LINE-Degraded
clean_val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

line_val_transforms = transforms.Compose([
    SimulateLINECompression(quality=70, max_size=1080), # LINE simulation
    transforms.Resize((224, 224)), # Resize for the model
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets (using 500 images for a quick test)
clean_val_data = GenImageDataset(dataset['validation'].select(range(500)), clean_val_transforms)
line_val_data = GenImageDataset(dataset['validation'].select(range(500)), line_val_transforms)

clean_val_loader = DataLoader(clean_val_data, batch_size=32, shuffle=False)
line_val_loader = DataLoader(line_val_data, batch_size=32, shuffle=False)

# ==========================================
# 2. LOAD YOUR SAVED MODEL
# ==========================================
print("Loading saved Baseline Model...")
model = models.resnet50(weights=None)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model.load_state_dict(torch.load('baseline_ai_detector.pth'))
model = model.to(device)
model.eval()

# ==========================================
# 3. EVALUATION FUNCTION
# ==========================================
def evaluate_model(dataloader, description):
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=description):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100. * correct / total
    return accuracy

# ==========================================
# 4. RUN THE COMPARISON
# ==========================================
print("\n--- Running Thesis Baseline Test ---")
clean_acc = evaluate_model(clean_val_loader, "Testing Clean Images")
line_acc = evaluate_model(line_val_loader, "Testing LINE-Degraded Images")

print("\n=== RESULTS ===")
print(f"Accuracy on Clean Images:      {clean_acc:.2f}%")
print(f"Accuracy on LINE Images:       {line_acc:.2f}%")
print(f"Accuracy Drop (Degradation):   {(clean_acc - line_acc):.2f}%")

Loading saved Baseline Model...

--- Running Thesis Baseline Test ---


Testing LINE-Degraded Images: 100%|██████████| 16/16 [02:25<00:00,  9.12s/it]


=== RESULTS ===
Accuracy on Clean Images:      76.60%
Accuracy on LINE Images:       73.20%
Accuracy Drop (Degradation):   3.40%
